In [6]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, regularizers
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.preprocessing import image

In [8]:
BASE_DIR = os.path.join(os.getcwd(), 'Datos')
TRAIN_DIR = os.path.join(BASE_DIR, 'Train')
VAL_DIR = os.path.join(BASE_DIR, 'Validation')

print('Directorio de trabajo:', os.getcwd())
print('BASE_DIR existe:', os.path.exists(BASE_DIR))
print('TRAIN_DIR existe:', os.path.exists(TRAIN_DIR))
print('VAL_DIR existe:', os.path.exists(VAL_DIR))

train_datagen = ImageDataGenerator(
    rescale=1./255, rotation_range=40, width_shift_range=0.2,
    height_shift_range=0.2, shear_range=0.2, zoom_range=0.2,
    horizontal_flip=True, fill_mode='nearest'
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(TRAIN_DIR, target_size=(150, 150), batch_size=32, class_mode='binary')
val_generator = val_datagen.flow_from_directory(VAL_DIR, target_size=(150, 150), batch_size=32, class_mode='binary')
print("Mapeo de clases:", train_generator.class_indices)


Directorio de trabajo: c:\Visual\Frameworks_IA\Detector_santa
BASE_DIR existe: True
TRAIN_DIR existe: True
VAL_DIR existe: True
Found 37 images belonging to 2 classes.
Found 37 images belonging to 2 classes.
Mapeo de clases: {'No_santa': 0, 'Santa': 1}


In [11]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Flatten(),
    layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(train_generator, epochs=20, validation_data=val_generator)
model.save('detector_santa_model.h5')

Epoch 1/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 658ms/step - accuracy: 0.5405 - loss: 1.7643 - val_accuracy: 0.4324 - val_loss: 1.6000
Epoch 2/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 664ms/step - accuracy: 0.5135 - loss: 1.6965 - val_accuracy: 0.5676 - val_loss: 1.3564
Epoch 3/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 680ms/step - accuracy: 0.4865 - loss: 1.4169 - val_accuracy: 0.4324 - val_loss: 1.2737
Epoch 4/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 655ms/step - accuracy: 0.6216 - loss: 1.2599 - val_accuracy: 0.5946 - val_loss: 1.1987
Epoch 5/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 658ms/step - accuracy: 0.5676 - loss: 1.2303 - val_accuracy: 0.5676 - val_loss: 1.1934
Epoch 6/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 357ms/step - accuracy: 0.5676 - loss: 1.1969 - val_accuracy: 0.6486 - val_loss: 1.1188
Epoch 7/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 662ms/step - accuracy: 0.4865 - loss: 1.1550 - val_accuracy: 0.4324 - val_loss: 1.1661
Epoch 8/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 363ms/step - accuracy: 0.4324 - loss: 1.1611 - val_accuracy: 0.6757 - val_loss:

In [14]:
def clasificar_imagen(ruta_imagen):
    img = image.load_img(ruta_imagen, target_size=(150, 150))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0

    probabilidad = model.predict(img_array)[0][0]
    if probabilidad > 0.5:
        print(f"¡Es SANTA Claus! 🎅 ({probabilidad*100:.2f}%)")
    else:
        print(f"No es Santa ❌ ({(1 - probabilidad)*100:.2f}%)")

# Para probar una foto, solo cambia el nombre aquí y corre esta celda:
clasificar_imagen('Captura de pantalla 2026-06-20 183448.png')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
No es Santa ❌ (89.07%)
